In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "50")

In [0]:
df = spark.read.csv("/Volumes/external-catalog/default/test-volume/Employee_Attrition.csv", header=True, inferSchema=True)
display(df)

In [0]:
from pyspark.sql.functions import col

# Filter high risk attrition employees: Attrition = 'No' and JobSatisfaction < 3
high_risk_df = df.filter(
    (col("Attrition") == "No") & (col("JobSatisfaction") < 3)
)

# Select relevant informative columns
high_risk_selected = high_risk_df.select(
    "EmployeeNumber",
    "Age",
    "Gender",
    "MaritalStatus",
    "Department",
    "JobRole",
    "JobLevel",
    "JobSatisfaction",
    "EnvironmentSatisfaction",
    "WorkLifeBalance",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "MonthlyIncome",
    "OverTime",
    "DistanceFromHome"
)

display(high_risk_selected)

# Write to Delta table in external-catalog.default schema
high_risk_selected.write.format("delta").mode("overwrite").saveAsTable(
    "`external-catalog`.default.high_risk_attrition_employees"
)

In [0]:
# Read Employee Attrition data from CSV in the external-catalog volume
attrition_df = spark.read.csv("/Volumes/external-catalog/default/test-volume/Employee_Attrition.csv", header=True, inferSchema=True)

# Write to Delta table in external-catalog.default schema
attrition_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.default.employee_attrition")

print("Table 'external-catalog.default.employee_attrition' created successfully.")
display(spark.table("`external-catalog`.default.employee_attrition"))

In [0]:
# List Delta table versions for both tables
print("=== employee_attrition table history ===")
display(spark.sql("DESCRIBE HISTORY `external-catalog`.default.employee_attrition"))

print("=== high_risk_attrition_employees table history ===")
display(spark.sql("DESCRIBE HISTORY `external-catalog`.default.high_risk_attrition_employees"))

In [0]:
# Read the Delta table for high risk attrition employees
high_risk_df = spark.table("`external-catalog`.default.high_risk_attrition_employees")
display(high_risk_df)

In [0]:
from pyspark.sql import Row

# Insert 1 dummy record into the high_risk_attrition_employees Delta table
dummy_record = spark.createDataFrame([Row(
    EmployeeNumber=99999,
    Age=30,
    Gender="Male",
    MaritalStatus="Single",
    Department="Research & Development",
    JobRole="Laboratory Technician",
    JobLevel=1,
    JobSatisfaction=1,
    EnvironmentSatisfaction=2,
    WorkLifeBalance=2,
    YearsAtCompany=3,
    YearsInCurrentRole=2,
    YearsSinceLastPromotion=1,
    MonthlyIncome=3000,
    OverTime="Yes",
    DistanceFromHome=10
)])

dummy_record.write.format("delta").mode("append").saveAsTable(
    "`external-catalog`.default.high_risk_attrition_employees"
)

print("Dummy record inserted successfully.")
display(spark.table("`external-catalog`.default.high_risk_attrition_employees").filter("EmployeeNumber = 99999"))

In [0]:
# List all versions of the high_risk_attrition_employees Delta table
display(spark.sql("DESCRIBE HISTORY `external-catalog`.default.high_risk_attrition_employees"))

In [0]:
# Read the high_risk_attrition_employees Delta table
high_risk_df = spark.table("`external-catalog`.default.high_risk_attrition_employees")
display(high_risk_df)

In [0]:
# Read the high_risk_attrition_employees Delta table at version 0
high_risk_version_df = spark.read.option("versionAsOf", 0).table("`external-catalog`.default.high_risk_attrition_employees")
display(high_risk_version_df)

In [0]:
# Read the high_risk_attrition_employees Delta table as of a specific timestamp
high_risk_timestamp_df = spark.read.option("timestampAsOf", "2026-10-15T01:56:38").table("`external-catalog`.default.high_risk_attrition_employees")
display(high_risk_timestamp_df)

In [0]:
# Create a managed volume named 'employee_transformed_data' in external-catalog.default schema
spark.sql("CREATE VOLUME IF NOT EXISTS `external-catalog`.default.employee_transformed_data COMMENT 'Managed volume for storing transformed employee data'")
print("Managed volume 'employee_transformed_data' created successfully in external-catalog.default.")

# Verify the volume was created
display(spark.sql("SHOW VOLUMES IN `external-catalog`.default"))

In [0]:
from pyspark.sql.functions import col, when, lit, concat, round as spark_round

# Read the employee attrition table
emp_df = spark.table("`external-catalog`.default.employee_attrition")

# Apply logical transformations
transformed_df = emp_df \
    .withColumn(
        "AttritionRisk",
        when((col("JobSatisfaction") < 3) & (col("OverTime") == "Yes"), "High")
        .when((col("JobSatisfaction") < 3) | (col("OverTime") == "Yes"), "Medium")
        .otherwise("Low")
    ) \
    .withColumn(
        "ExperienceLevel",
        when(col("TotalWorkingYears") < 5, "Junior")
        .when((col("TotalWorkingYears") >= 5) & (col("TotalWorkingYears") < 15), "Mid")
        .otherwise("Senior")
    ) \
    .withColumn(
        "IncomeCategory",
        when(col("MonthlyIncome") < 3000, "Low")
        .when((col("MonthlyIncome") >= 3000) & (col("MonthlyIncome") < 8000), "Medium")
        .otherwise("High")
    ) \
    .withColumn(
        "TenureStability",
        spark_round(col("YearsAtCompany") / (col("NumCompaniesWorked") + 1), 2)
    ) \
    .withColumn(
        "PromotionOverdue",
        when(col("YearsSinceLastPromotion") > 5, True).otherwise(False)
    ) \
    .withColumn(
        "EmployeeTag",
        concat(col("Department"), lit("_"), col("JobRole"), lit("_"), col("EmployeeNumber"))
    )

print(f"Total records: {transformed_df.count()}")
print(f"Departments: {[row.Department for row in transformed_df.select('Department').distinct().collect()]}")
display(transformed_df.select(
    "EmployeeNumber", "EmployeeTag", "Department", "JobRole", "Age",
    "MonthlyIncome", "IncomeCategory", "ExperienceLevel", "AttritionRisk",
    "TenureStability", "PromotionOverdue", "OverTime", "Attrition"
))

# Write to managed volume partitioned by Department
output_path = "/Volumes/external-catalog/default/employee_transformed_data/partitioned_by_department"

transformed_df.write \
    .partitionBy("Department") \
    .mode("overwrite") \
    .parquet(output_path)

print(f"\nTransformed data written to: {output_path}")
print("Partitioned by Department.")

# Verify the written partitions
display(dbutils.fs.ls(output_path))